## Project: Question-Answering on Private Documents

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

In [ ]:
def load_document(file):
    import os
    name, extension = os.path.splitext(file)
    if extension == '.pdf':
        from langchain_community.document_loaders import PyPDFLoader
        print(f"Loading {file}...")
        loader = PyPDFLoader(file)
    elif extension == '.txt':
        from langchain_community.document_loaders import TextLoader
        print(f"Loading {file}...")
        loader = TextLoader(file)
    elif extension == '.docx':
        from langchain_community.document_loaders import Docx2txtLoader
        print(f"Loading {file}...")
        loader = Docx2txtLoader(file)
    else:
        print(f"Unsupported file extension: {extension}")
        return None
    
    data = loader.load()
    return data

In [ ]:
data = load_document('files/us_constitution.pdf')

In [ ]:
print(data[1].page_content)

In [ ]:
data = load_document('files/the_great_gatsby.docx')
print(data[0].page_content)

In [ ]:
# Loading from a website

def load_from_wikipedia(query, lang="en", num_pages=2):
    from langchain_community.document_loaders import WikipediaLoader
    loader = WikipediaLoader(query=query, lang=lang,load_max_docs=num_pages)
    data = loader.load()
    return data

In [ ]:
data = load_from_wikipedia("GPT-4")
print(data[0].page_content)

#### Chunking the loaded documents

In [ ]:
# Loading US constitution
data = load_document('files/us_constitution.pdf')

from langchain_text_splitters import RecursiveCharacterTextSplitter
def chunk_data(data, chunk_size=256):
    print(f"Chunking the data into chunks of {chunk_size}...")
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size)
    chunks = text_splitter.split_documents(data)
    return chunks

In [ ]:
chunks = chunk_data(data)
chunks[0].page_content

In [ ]:
# Calculating the chunks cost
def print_embedding_cost(texts):
    import tiktoken
    enc = tiktoken.encoding_for_model("text-embedding-ada-002")
    total_tokens = sum([len(enc.encode(page.page_content)) for page in texts])
    print(f"Total tokens: {total_tokens}")
    print(f"Cost: ${total_tokens / 1000 * 0.0004:.6f}")

In [ ]:
print_embedding_cost(chunks)

#### Pinecone

In [ ]:
from pinecone import Pinecone
from pinecone import ServerlessSpec
pc = Pinecone()

print(pc.list_indexes())

In [ ]:
index_name = 'langchain'
if index_name not in pc.list_indexes():
    print(f"Creating index {index_name}...😊")
    pc.create_index(index_name, 
                    dimension=1536,
                    metric='cosine',
                    spec=ServerlessSpec(cloud='aws', region='us-east-1'))
    

In [ ]:
index_name="langchain"
pc.delete_index(index_name)

In [ ]:
index = pc.Index(index_name)
print(index.describe_index_stats())

### Working with Vectors

In [ ]:
# Create a dummy vector
import random
vectors = [[random.random() for _ in range(1536)] for _ in range(6)]
print(vectors)
print(len(vectors))


In [ ]:
# To insert vector into a pinecone index we always need a unique id for each vector
ids = list('ABCDEF')
print(ids)

In [ ]:
# Insert Vectors into Pinecone
index = pc.Index(index_name) # index_name is 'langchain' in this case.
index.upsert(vectors=zip(ids, vectors))

In [ ]:
# Update Vectors
index.upsert(vectors=[('C', [0.5] * 1536)])

In [ ]:
# Fetch teh vectors using ids
index.fetch(ids = ['C'])

In [ ]:
# Delete Vectors using ids
index.delete(ids = ['A', 'C'])

In [ ]:
pc.delete_index(index_name)

In [ ]:
# Querying the vectors using Pinecone
X = [random.random() for _ in range(1536)]

index.query(
    vector=X,
    top_k=2,
    include_values=False
)


### Namespace

Pinecone allows you to group the vectors in an index using a namespace.  
You can insert vectors into a particular namespace.

In [ ]:
# Create a dummy vectors list to insert into a namespace
first_namespace_vector = [[random.random() for _ in range(1536)] for _ in range(3)]
ids = list('XYZ')
index.upsert(vectors=zip(ids, first_namespace_vector), namespace="first-namespace")

In [ ]:
print(index.describe_index_stats())

In [ ]:
second_namespace_vectors = [[random.random() for _ in range(1536)] for _ in range(2)]
ids = list('qp')
index.upsert(vectors=zip(ids, second_namespace_vectors), namespace="second-namespace")

In [ ]:
print(index.describe_index_stats())

In [ ]:
# Now if we try to fetch without namespace we wont get the vector 
index.fetch(ids=['X'])
# Returns empty vector since namespace is not specified

In [ ]:
index.fetch(ids=['X'], namespace="first-namespace")
# Return the vector since the namespace is specified

In [ ]:
index.delete(ids=['X'], namespace="first-namespace")
# Delete works with the same logic.

In [ ]:
# Delete everything in a namespace 
index.delete(delete_all=True, namespace='first-namespace')
